# Model Comparison

Compare LightGBM vs XGBoost on the loan risk dataset.
Run the training pipeline first: `make train`

In [ ]:
import mlflow
import pandas as pd
import matplotlib.pyplot as plt
from loan_risk.config import get_settings

cfg = get_settings()
mlflow.set_tracking_uri(cfg.mlflow.tracking_uri)

In [ ]:
# Load all runs from the experiment
runs = mlflow.search_runs(
    experiment_names=[cfg.mlflow.experiment_name],
    order_by=['metrics.test_auc_roc DESC'],
)
print(f'Total runs: {len(runs)}')
runs[['params.model_name', 'metrics.test_auc_roc', 'metrics.val_auc_roc',
      'metrics.test_ks_statistic', 'metrics.test_gini']].head(10)

In [ ]:
# Compare test AUC across model types
if len(runs) > 0 and 'params.model_name' in runs.columns:
    model_comparison = runs.groupby('params.model_name')['metrics.test_auc_roc'].agg(['mean', 'max', 'std'])
    print('AUC by Model Type:')
    print(model_comparison)
    
    model_comparison['max'].plot(kind='bar', title='Best Test AUC by Model Type', figsize=(8, 4))
    plt.ylabel('AUC-ROC')
    plt.axhline(0.80, color='red', linestyle='--', label='Promotion gate')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No completed runs found. Run make train first.')